# Microsoft Agent Framework — Azure OpenAI (Responses API)

V tomto ukázkovém kódu použijete **Microsoft Agent Framework (MAF)** k vytvoření jednoduchého agenta podporovaného **Azure OpenAI** pomocí **Responses API**.

> **Poznámka k migraci:** Tento příklad dříve používal Semantic Kernel s GitHub Models. Byl migrován do Microsoft Agent Framework a GitHub Models (zastaralý, plánované ukončení v červenci 2026) byl nahrazen Azure OpenAI, které podporuje Responses API. `OpenAIChatClient` v MAF cílí na stabilní endpoint Azure OpenAI `/openai/v1/` a ve výchozím nastavení používá Responses API.

Cílem tohoto příkladu je ukázat kroky, které budou později použity v dalších ukázkových kódech při implementaci různých agentních vzorů.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importujte potřebné balíčky Pythonu


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definování nástroje

V rámci Microsoft Agent Framework je **nástroj** obyčejná Python funkce dekorovaná `@tool`, kterou může agent zavolat. Níže definujeme nástroj, který vrací náhodné dovolenkové destinace a vyhýbá se opakování té předchozí.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Vytvoření agenta

Zde vytvoříme agenta jménem `TravelAgent`.

V tomto příkladu používáme velmi základní instrukce. Klidně tyto instrukce upravte a sledujte, jak se změní chování agenta.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Spuštění agenta

Nyní můžeme agenta spustit. Vytvoříme `AgentSession`, aby si agent pamatoval konverzaci přes jednotlivé kroky, a pak pošleme dva `user_inputs`. První žádá o výlet; druhý říká, že se uživateli návrh nelíbil a žádá o jiný — agent využívá historii sezení plus nástroj `get_random_destination`, aby odpověděl.

Můžete upravit tyto zprávy, abyste pozorovali, jak agent reaguje různě. Odpovědi jsou **streamovány** token po tokenu.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
